# Silver — `ecommerce_pedidos`

Este notebook lê micro-lotes da Bronze, aplica as 10 regras de qualidade da tabela de pedidos, grava a Silver em Delta e registra os resultados em `squad1.dq_monitoring_logs`.



In [0]:
#**Entrada principal:**
#- `squad1.bronze_ecommerce_pedidos`

#**Saídas:**
#- `squad1.silver_ecommerce_pedidos`
#- `squad1.dq_monitoring_logs`

#A Silver mantém todas as linhas do micro-lote e adiciona flags booleanas indicando falhas por regra.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
import uuid

run_id = str(uuid.uuid4())

BRONZE_PEDIDOS_TABLE = "squad1.bronze_ecommerce_pedidos"
SILVER_PEDIDOS_TABLE = "squad1.silver_ecommerce_pedidos"
DQ_LOGS_TABLE = "squad1.dq_monitoring_logs"

print("run_id:", run_id)



## Funções auxiliares

In [0]:
def tabela_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def resolver_tabela(possiveis_tabelas):
    """Retorna a primeira tabela existente entre as opções informadas."""
    for tabela in possiveis_tabelas:
        if tabela_existe(tabela):
            print(f"Tabela encontrada: {tabela}")
            return tabela
    raise Exception(f"Nenhuma das tabelas existe: {possiveis_tabelas}")


def garantir_tabela_dq_logs():
    schema_dq_logs = StructType([
        StructField("run_id", StringType(), False),
        StructField("tabela", StringType(), False),
        StructField("regra", StringType(), False),
        StructField("status", StringType(), False),
        StructField("severidade", StringType(), False),
        StructField("qtd_registros_falhos", IntegerType(), False),
        StructField("qtd_registros_total", IntegerType(), False),
        StructField("timestamp_execucao", TimestampType(), False),
        StructField("arquivo_origem", StringType(), False),
    ])

    if not tabela_existe(DQ_LOGS_TABLE):
        df_empty = spark.createDataFrame([], schema_dq_logs)

        (
            df_empty.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(DQ_LOGS_TABLE)
        )

        print(f"Tabela criada: {DQ_LOGS_TABLE}")
    else:
        print(f"Tabela já existe: {DQ_LOGS_TABLE}")


def criar_log_regra(df, nome_tabela, nome_regra, coluna_flag, severidade):
    return (
        df
        .groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(
                F.when(F.col(coluna_flag) == True, 1).otherwise(0)
            ).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(run_id))
        .withColumn("tabela", F.lit(nome_tabela))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn(
            "status",
            F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL"))
             .otherwise(F.lit("PASS"))
        )
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id",
            "tabela",
            "regra",
            "status",
            "severidade",
            "qtd_registros_falhos",
            "qtd_registros_total",
            "timestamp_execucao",
            "arquivo_origem"
        )
    )


def salvar_delta_append_sem_duplicar_logs(df_logs):
    """Evita duplicidade em dq_monitoring_logs por tabela + regra + arquivo_origem."""
    garantir_tabela_dq_logs()

    df_existente = (
        spark.table(DQ_LOGS_TABLE)
        .select("tabela", "regra", "arquivo_origem")
        .dropDuplicates()
    )

    df_novos = (
        df_logs
        .join(
            df_existente,
            on=["tabela", "regra", "arquivo_origem"],
            how="left_anti"
        )
    )

    qtd_novos = df_novos.count()

    if qtd_novos == 0:
        print("Nenhum log novo para gravar em dq_monitoring_logs.")
        return

    (
        df_novos.write
        .format("delta")
        .mode("append")
        .saveAsTable(DQ_LOGS_TABLE)
    )

    print(f"Logs gravados em {DQ_LOGS_TABLE}: {qtd_novos}")


## Ler Bronze de pedidos e selecionar apenas micro-lotes novos

In [0]:

if not tabela_existe(BRONZE_PEDIDOS_TABLE):
    raise Exception(f"Tabela Bronze não encontrada: {BRONZE_PEDIDOS_TABLE}")

df_bronze_pedidos = spark.table(BRONZE_PEDIDOS_TABLE)

if "bronze_source_file" not in df_bronze_pedidos.columns:
    raise Exception("A Bronze precisa possuir a coluna bronze_source_file.")

if tabela_existe(SILVER_PEDIDOS_TABLE):
    arquivos_ja_processados = (
        spark.table(SILVER_PEDIDOS_TABLE)
        .select("bronze_source_file")
        .dropDuplicates()
    )

    df_pedidos_lote = (
        df_bronze_pedidos
        .join(arquivos_ja_processados, on="bronze_source_file", how="left_anti")
    )
else:
    df_pedidos_lote = df_bronze_pedidos

qtd_lote = df_pedidos_lote.count()
TEM_MICRO_LOTE_NOVO = qtd_lote > 0

print(f"Registros novos para processar na Silver: {qtd_lote}")

if not TEM_MICRO_LOTE_NOVO:
    print("Nenhum micro-lote novo para processar. O notebook continuará para permitir validações.")
else:
    display(
        df_pedidos_lote
        .select("bronze_source_file")
        .dropDuplicates()
        .orderBy("bronze_source_file")
    )


## Ler tabelas de referência

In [0]:

# Referências usadas nas regras:
# - endereços: R10
# - rastreamento: R8

TABELA_ENDERECOS = resolver_tabela([
    "squad1.silver_ecommerce_enderecos",
    "squad1.bronze_ecommerce_enderecos"
])

TABELA_RASTREAMENTO = resolver_tabela([
    "squad1.silver_ecommerce_rastreamento_entregas",
    "squad1.silver_ecommerce_rastreamento",
    "squad1.bronze_ecommerce_rastreamento_entregas",
    "squad1.bronze_ecommerce_rastreamento"
])

df_enderecos_ref = spark.table(TABELA_ENDERECOS)
df_rastreamento_ref = spark.table(TABELA_RASTREAMENTO)

print("Tabela de endereços usada:", TABELA_ENDERECOS)
print("Tabela de rastreamento usada:", TABELA_RASTREAMENTO)


##  Padronização mínima de tipos para validação

In [0]:
df_base = (
    df_pedidos_lote
    .withColumn("id_pedido", F.col("id_pedido").cast("string"))
    .withColumn("id_cliente", F.col("id_cliente").cast("string"))
    .withColumn("id_endereco_entrega", F.col("id_endereco_entrega").cast("string"))
    .withColumn("status_pedido", F.trim(F.col("status_pedido")))
    .withColumn("metodo_pagamento", F.trim(F.col("metodo_pagamento")))
    .withColumn("valor_total", F.col("valor_total").cast("double"))
    .withColumn("valor_frete", F.col("valor_frete").cast("double"))
    .withColumn("dt_pedido", F.to_timestamp(F.col("dt_pedido")))
    .withColumn("dt_ultima_atualizacao_status", F.to_timestamp(F.col("dt_ultima_atualizacao_status")))
)


##  Preparar referências auxiliares

In [0]:

# R10 — Endereço de entrega existente
df_enderecos_validos = (
    df_enderecos_ref
    .select(F.col("id_endereco").cast("string").alias("id_endereco_entrega"))
    .dropDuplicates()
    .withColumn("endereco_entrega_existe", F.lit(True))
)

# R8 — Rastreamento ativo para pedidos cancelados
colunas_rastreamento = set(df_rastreamento_ref.columns)

if "id_pedido_ecommerce" in colunas_rastreamento:
    coluna_pedido_rastreamento = "id_pedido_ecommerce"
elif "id_pedido" in colunas_rastreamento:
    coluna_pedido_rastreamento = "id_pedido"
else:
    raise Exception(
        "A tabela de rastreamento precisa conter 'id_pedido_ecommerce' ou 'id_pedido'. "
        f"Colunas encontradas: {df_rastreamento_ref.columns}"
    )

print(f"Coluna de pedido usada no rastreamento: {coluna_pedido_rastreamento}")

df_rastreamento_base = (
    df_rastreamento_ref
    .withColumn("id_pedido", F.col(coluna_pedido_rastreamento).cast("string"))
)

possiveis_colunas_status_rastreamento = [
    "status_entrega",
    "status_rastreamento",
    "status",
    "status_logistico"
]

coluna_status_rastreamento = None

for c in possiveis_colunas_status_rastreamento:
    if c in colunas_rastreamento:
        coluna_status_rastreamento = c
        break

if coluna_status_rastreamento:
    print(f"Coluna de status de rastreamento usada: {coluna_status_rastreamento}")

    # Considera ativo qualquer evento de rastreamento que ainda não esteja entregue/finalizado.
    df_rastreamento_ativo = (
        df_rastreamento_base
        .withColumn(
            "status_rastreamento_aux",
            F.lower(F.trim(F.col(coluna_status_rastreamento)))
        )
        .filter(~F.col("status_rastreamento_aux").isin("entregue", "finalizado"))
        .select("id_pedido")
        .dropDuplicates()
        .withColumn("tem_rastreamento_ativo", F.lit(True))
    )
else:
    print("Nenhuma coluna de status de rastreamento encontrada. Qualquer registro de rastreamento será considerado ativo.")

    df_rastreamento_ativo = (
        df_rastreamento_base
        .select("id_pedido")
        .dropDuplicates()
        .withColumn("tem_rastreamento_ativo", F.lit(True))
    )

print("Schema df_rastreamento_ativo:")
df_rastreamento_ativo.printSchema()


##  Aplicar as 10 regras de qualidade

In [0]:

status_validos = [
    "Processando",
    "Pagamento Aprovado",
    "Em Separação",
    "Enviado",
    "Entregue",
    "Cancelado"
]

metodos_pagamento_validos = [
    "Cartão de Crédito",
    "Pix",
    "Boleto"
]

w_id_pedido = Window.partitionBy("id_pedido")

df_regras = (
    df_base
    .withColumn("qtd_id_pedido", F.count("*").over(w_id_pedido))
    .join(df_rastreamento_ativo, on="id_pedido", how="left")
    .join(df_enderecos_validos, on="id_endereco_entrega", how="left")
    .withColumn("tem_rastreamento_ativo", F.coalesce(F.col("tem_rastreamento_ativo"), F.lit(False)))
    .withColumn("endereco_entrega_existe", F.coalesce(F.col("endereco_entrega_existe"), F.lit(False)))
)

df_silver_pedidos = (
    df_regras
    # R1 — id_pedido não pode ser nulo nem duplicado
    .withColumn(
        "r1_id_pedido_falhou",
        F.col("id_pedido").isNull() | (F.col("qtd_id_pedido") > 1)
    )
    # R2 — id_cliente não pode ser nulo
    .withColumn(
        "r2_id_cliente_falhou",
        F.col("id_cliente").isNull()
    )
    # R3 — status_pedido deve estar na lista de valores permitidos
    .withColumn(
        "r3_status_pedido_falhou",
        F.col("status_pedido").isNull() | (~F.col("status_pedido").isin(status_validos))
    )
    # R4 — valor_total deve ser > 0 e numérico
    .withColumn(
        "r4_valor_total_falhou",
        F.col("valor_total").isNull() | (F.col("valor_total") <= 0)
    )
    # R5 — metodo_pagamento deve estar na lista de valores permitidos
    .withColumn(
        "r5_metodo_pagamento_falhou",
        F.col("metodo_pagamento").isNull() | (~F.col("metodo_pagamento").isin(metodos_pagamento_validos))
    )
    # R6 — dt_ultima_atualizacao_status não pode ser anterior a dt_pedido
    .withColumn(
        "r6_dt_status_anterior_pedido_falhou",
        F.col("dt_ultima_atualizacao_status").isNotNull()
        & F.col("dt_pedido").isNotNull()
        & (F.col("dt_ultima_atualizacao_status") < F.col("dt_pedido"))
    )
    # R7 — valor_frete deve ser 0 quando valor_total >= 250
    .withColumn(
        "r7_frete_gratis_acima_250_falhou",
        (F.col("valor_total") >= F.lit(250.0))
        & (
            F.col("valor_frete").isNull()
            | (F.col("valor_frete") != F.lit(0.0))
        )
    )
    # R8 — pedidos cancelados não devem ter registros de rastreamento ativos
    .withColumn(
        "r8_cancelado_com_rastreamento_ativo_falhou",
        (F.col("status_pedido") == "Cancelado") & (F.col("tem_rastreamento_ativo") == True)
    )
    # R9 — pedidos entregues devem ter dt_ultima_atualizacao_status >= dt_pedido + 2 dias
    .withColumn(
        "r9_entregue_data_invalida_falhou",
        (F.col("status_pedido") == "Entregue")
        & F.col("dt_pedido").isNotNull()
        & F.col("dt_ultima_atualizacao_status").isNotNull()
        & (F.col("dt_ultima_atualizacao_status") < F.expr("dt_pedido + INTERVAL 2 DAYS"))
    )
    # R10 — id_endereco_entrega deve existir em ecommerce_enderecos
    .withColumn(
        "r10_endereco_entrega_inexistente_falhou",
        F.col("id_endereco_entrega").isNull() | (F.col("endereco_entrega_existe") == False)
    )
    .withColumn(
        "silver_linha_valida",
        ~(
            F.col("r1_id_pedido_falhou")
            | F.col("r2_id_cliente_falhou")
            | F.col("r3_status_pedido_falhou")
            | F.col("r4_valor_total_falhou")
            | F.col("r5_metodo_pagamento_falhou")
            | F.col("r6_dt_status_anterior_pedido_falhou")
            | F.col("r7_frete_gratis_acima_250_falhou")
            | F.col("r8_cancelado_com_rastreamento_ativo_falhou")
            | F.col("r9_entregue_data_invalida_falhou")
            | F.col("r10_endereco_entrega_inexistente_falhou")
        )
    )
    .withColumn("silver_processed_at", F.current_timestamp())
    .drop("qtd_id_pedido")
)

display(df_silver_pedidos.limit(10))


## Gravar Silver em Delta

In [0]:

qtd_silver = df_silver_pedidos.count()

if qtd_silver == 0:
    print("Nenhum registro novo para gravar na Silver de pedidos.")
else:
    (
        df_silver_pedidos.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_PEDIDOS_TABLE)
    )

    print(f"Silver gravada em: {SILVER_PEDIDOS_TABLE}")
    print(f"Registros gravados: {qtd_silver}")


##  Criar logs de qualidade em `squad1.dq_monitoring_logs`

In [0]:
df_logs = (
    criar_log_regra(
        df_silver_pedidos,
        "silver_ecommerce_pedidos",
        "R1 - id_pedido não pode ser nulo nem duplicado",
        "r1_id_pedido_falhou",
        "Critica"
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R2 - id_cliente não pode ser nulo",
            "r2_id_cliente_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R3 - status_pedido deve estar na lista permitida",
            "r3_status_pedido_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R4 - valor_total deve ser maior que zero e numérico",
            "r4_valor_total_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R5 - metodo_pagamento deve estar na lista permitida",
            "r5_metodo_pagamento_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R6 - dt_ultima_atualizacao_status não pode ser anterior a dt_pedido",
            "r6_dt_status_anterior_pedido_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R7 - valor_frete deve ser zero quando valor_total maior ou igual a 250",
            "r7_frete_gratis_acima_250_falhou",
            "Critica"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R8 - pedido cancelado não deve ter rastreamento ativo",
            "r8_cancelado_com_rastreamento_ativo_falhou",
            "Aviso"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R9 - pedido entregue deve ter atualização ao menos 2 dias após o pedido",
            "r9_entregue_data_invalida_falhou",
            "Aviso"
        )
    )
    .unionByName(
        criar_log_regra(
            df_silver_pedidos,
            "silver_ecommerce_pedidos",
            "R10 - id_endereco_entrega deve existir em ecommerce_enderecos",
            "r10_endereco_entrega_inexistente_falhou",
            "Critica"
        )
    )
)

display(df_logs)

salvar_delta_append_sem_duplicar_logs(df_logs)


##  Validação final

In [0]:
display(
    df_silver_pedidos
    .groupBy("silver_linha_valida")
    .count()
)

display(
    spark.table(DQ_LOGS_TABLE)
    .filter(F.col("tabela") == "silver_ecommerce_pedidos")
    .orderBy(F.col("timestamp_execucao").desc())
    .limit(50)
)

print("Processamento Silver de pedidos concluído.")
